
# 4.6 系统分析与加速

这一节把视角从“怎么写 kernel”推进到“怎么分析和复用 kernel 的执行”。

前面的章节主要回答“算子怎么写、怎么组合”。这一节转向系统视角：同样的 Softmax，既可以拿来观察模拟执行成本，也可以拿来观察图捕获与重放流程。

所以这里的重点不是再发明新算子，而是看清一个算子从 PyPTO kernel 走到 Cost Model、再走到 ACLGraph 时，分别需要补哪些包装、验证和运行约束。



## 1. 环境准备

Cost Model 可以在 SIM 模式下观察输出文件；ACLGraph 则需要真实 NPU 和 `torch_npu`。Notebook 里保留了完整写法；如果当前环境没有 NPU，可以先重点理解 ACLGraph 的代码链路。

这一节只要能跑 Cost Model，就能完整观察模拟输出。ACLGraph 部分则重点看代码链路、FakeTensor 分支和 graph capture / replay 的时序。


In [ ]:
import os
os.environ['TILE_FWK_DEVICE_ID'] = '0'
import json
import torch
import pypto
import numpy as np
from numpy.testing import assert_allclose
import torch_npu


try:
    from torch._dynamo import allow_in_graph
    from torch._subclasses.fake_tensor import FakeTensor
except Exception:
    allow_in_graph = None
    FakeTensor = None


def get_device():
    device_id = int(os.environ.get("TILE_FWK_DEVICE_ID", "0"))
    torch.npu.set_device(device_id)
    return f"npu:{device_id}"


def current_device(device_id=None):
    return f"npu:{device_id}"


device = get_device()
RUN_MODE = pypto.RunMode.NPU if device != "cpu" else pypto.RunMode.SIM

print("TILE_FWK_DEVICE_ID:", os.environ.get("TILE_FWK_DEVICE_ID", "<not set>"))
print("device:", device)
print("run_mode:", RUN_MODE)
print("pypto:", pypto.__file__)



## 2. 学完后你应该能够

学完这一节后，可以回到这里检查自己是否已经做到：

1. 说明 Cost Model 关注的不是单纯数值结果，而是模拟执行成本和可视化分析结果。
2. 说明 `merged_swimlane.json` 这类文件用于观察任务调度、硬件单元占用和执行路径。
3. 说明 ACLGraph 的目标是捕获并复用执行图，减少多次执行时的 Host 侧开销。
4. 看懂 `@allow_in_graph` 和 `FakeTensor` 为什么会出现在 PyTorch 集成场景里。
5. 区分“算子数学正确性验证”和“系统行为 / 图捕获验证”这两类检查。



## 3. 这一节会依次练什么

| 练习场景 | 位置 | 代表函数 | 关键点 |
| --- | --- | --- | --- |
| Cost Model Softmax | 4.1 - 4.2 | `safe_json_load`、`get_output_path`、`cost_softmax_core`、`cost_model_softmax`、`test_cost_model_softmax` | SIM 模式、tiling、输出文件验证 |
| ACLGraph Softmax | 5.1 - 5.2 | `acl_softmax_core`、`acl_softmax_kernel`、`acl_softmax`、`SoftmaxModule`、`test_softmax_capture` | `allow_in_graph`、FakeTensor、NPUGraph |

两个例子都用 Softmax，但观察角度完全不同：Cost Model 是“这个算子怎么被执行、代价在哪里”，ACLGraph 是“怎么把执行路径捕获下来并重复使用”。

如果把这节看成一张地图，那么 Cost Model 更像放大镜，帮助理解算子的执行形态；ACLGraph 更像缓存层，帮助减少重复调度开销。



## 4. Cost Model：从算子到执行成本

这里使用 Softmax，不是因为它数学上特殊，而是因为它同时包含 reduction、elementwise、loop 和输出回写，足够看清一个算子是如何被拆成执行步骤的。

这一部分先看结果，再看 kernel。

- 结果层：`test_cost_model_softmax` 会把 PyPTO 输出和 `torch.softmax(input_data, dim=3)` 对齐，确认数学结果没有被执行组织方式破坏。
- 文件层：`./output/CostModelSimulationOutput/merged_swimlane.json` 是否生成、是否能被 `safe_json_load` 正常读取。
- 结构层：`cost_model_softmax` 如何把 batch 维度切成 tile，如何在 loop 里处理每个 tile，如何用 `pypto.assemble` 把结果写回。

这一节的阅读顺序可以先从 4.1 看文件，再从 4.2 看 kernel。


In [ ]:
def safe_json_load(file_path):
    try:
        with open(file_path, "r", encoding="utf-8") as file:
            data = json.load(file)
        return data, None
    except FileNotFoundError:
        return None, "File not found"
    except json.JSONDecodeError as e:
        return None, f"Invalid json format: {e}"
    except PermissionError:
        return None, "Permission Error"
    except Exception as e:
        return None, f"Load json failed: {e}"


def get_output_path():
    out_path = "./output"
    if os.path.exists(out_path):
        subdirs = [os.path.join(out_path, d) for d in os.listdir(out_path)
                   if os.path.isdir(os.path.join(out_path, d))]
        if subdirs:
            return max(subdirs, key=os.path.getctime)
    return None


def cost_softmax_core(input_tensor: pypto.Tensor) -> pypto.Tensor:
    row_max = pypto.amax(input_tensor, dim=-1, keepdim=True)
    sub = pypto.sub(input_tensor, row_max)
    exp = pypto.exp(sub)
    esum = pypto.sum(exp, dim=-1, keepdim=True)
    return pypto.div(exp, esum)


In [ ]:
@pypto.frontend.jit(runtime_options={"run_mode": pypto.RunMode.SIM})
def cost_model_softmax(
    input_tensor: pypto.Tensor(),
    output_tensor: pypto.Tensor()):
    tensor_shape = input_tensor.shape
    b = tensor_shape[0]
    n1, n2, dim = tensor_shape[1:]
    tile_b = 1
    b_loop = b // tile_b

    pypto.set_vec_tile_shapes(1, 4, 1, 64)
    for idx in pypto.loop(b_loop):
        b_offset = idx * tile_b
        b_offset_end = (idx + 1) * tile_b
        input_view = input_tensor[b_offset:b_offset_end, :n1, :n2, :dim]
        softmax_out = cost_softmax_core(input_view)
        pypto.assemble(softmax_out, [b_offset, 0, 0, 0], output_tensor)


def test_cost_model_softmax(cost_model_enable=True):
    shape = (32, 32, 1, 256)
    input_data = torch.rand(shape, dtype=torch.float32)
    output_data = torch.empty(shape, dtype=torch.float32)
    cost_model_softmax(input_data, output_data)

    torch_softmax = torch.softmax(input_data, dim=3)
    max_diff = np.abs(output_data.cpu().numpy() - torch_softmax.cpu().numpy()).max()
    print(f"CostModel Softmax input shape: {input_data.shape}")
    print(f"CostModel Softmax output shape: {output_data.shape}")
    print(f"CostModel Softmax max diff: {max_diff:.6f}")

    output_path = get_output_path()
    print("latest output path:", output_path)
    if cost_model_enable and output_path is not None:
        swimlane_path = os.path.join(output_path, "CostModelSimulationOutput", "merged_swimlane.json")
        merged_swimlane, error = safe_json_load(swimlane_path)
        print("merged_swimlane loaded:", error is None)
        if error:
            print("merged_swimlane check:", error)
    return output_path



### 4.1 如何阅读 Cost Model 结果

Cost Model 的输出不是普通数值结果，而是分析文件。`merged_swimlane.json` 可以理解成一份执行过程记录，用来观察不同硬件单元、任务阶段和调度关系。

`test_cost_model_softmax` 里其实做了两层验证：先用 `torch.softmax` 检查数值，再去 `./output` 下找最新目录，最后读取 `CostModelSimulationOutput/merged_swimlane.json`。前者确认 Softmax 语义没有变，后者确认 cost model 的模拟结果确实生成了。

读这段代码时，可以先看两个辅助函数：

- `get_output_path()`：在 `./output` 下找最新的子目录，避免手工猜输出路径。
- `safe_json_load()`：安全读取 JSON，文件不存在、格式错误或权限错误时返回明确错误信息。

然后再看 `cost_model_softmax` 的执行链路：

1. `tensor_shape = input_tensor.shape`，先记录输入整体 shape。
2. `b = tensor_shape[0]`，把 batch 维拆成动态维度。
3. `n1, n2, dim = tensor_shape[1:]`，其余三个维度保持静态，方便观察固定布局下的执行成本。
4. `tile_b = 1`，每次处理一个 batch tile，便于把循环和输出分析拆开看。
5. `b_loop = b // tile_b`，将 batch 数量换算为循环次数。
6. `pypto.set_vec_tile_shapes(1, 4, 1, 64)`，设置向量 tile 形状，只影响执行组织，不改变数学结果。
7. `for idx in pypto.loop(b_loop):`，把 batch 维显式展开成 loop。
8. `input_view = input_tensor[b_offset:b_offset_end, :n1, :n2, :dim]`，取出当前 batch 的局部视图。
9. `softmax_out = cost_softmax_core(input_view)`，在局部视图上执行 Softmax。
10. `pypto.assemble(softmax_out, [b_offset, 0, 0, 0], output_tensor)`，把局部结果写回总输出。

`cost_softmax_core` 本身还是标准稳定写法：`amax -> sub -> exp -> sum -> div`。这里用 `keepdim=True` 是为了让广播对齐更直接，避免额外 reshape。

### 4.2 逐段拆解 `cost_model_softmax`

这段 kernel 的关键不是“把 Softmax 写出来”，而是“把 Softmax 写成可观察的执行片段”。所以它故意保留了 batch 切片、循环和 assemble，这样 cost model 才能把这些步骤分别记录下来。

`test_cost_model_softmax` 的最后两步也很重要：

- `max_diff` 用来和 `torch.softmax` 对比，说明输出数值一致。
- `merged_swimlane, error = safe_json_load(...)` 用来确认模拟输出真的生成了，而且 JSON 格式可读。

如果只看数值而不看文件，就会漏掉 cost model 的价值；如果只看文件而不看数值，就会失去算子正确性保障。


In [ ]:
test_cost_model_softmax()



## 5. ACLGraph：捕获执行图并复用

ACLGraph 关注的是另一类开销：Host 侧反复提交任务的成本。对会被多次执行的模型片段，可以先捕获一次执行图，再通过 `replay` 重放。

这个例子仍然使用 Softmax，但写法上多了 PyTorch 集成层：`@allow_in_graph`、`FakeTensor`、`torch.compile`、`torch.npu.NPUGraph`、`torch.npu.graph(g)` 和 `g.replay()`。

注意这里的目标不是改变 Softmax 的数学，而是改变“这段 Softmax 如何进入图、如何被捕获、如何被复用”。


In [ ]:
B = pypto.DYNAMIC
N1, N2, DIM = 32, 1, 256


def acl_softmax_core(x: pypto.Tensor) -> pypto.Tensor:
    row_max = pypto.amax(x, dim=-1, keepdim=True)
    sub = x - row_max
    exp = pypto.exp(sub)
    esum = pypto.sum(exp, dim=-1, keepdim=True)
    return exp / esum


@pypto.frontend.jit()
def acl_softmax_kernel(
    input_tensor: pypto.Tensor([B, N1, N2, DIM], pypto.DT_FP32),
    output_tensor: pypto.Tensor([B, N1, N2, DIM], pypto.DT_FP32)):
    bs = input_tensor.shape[0]
    tile_b = 1
    b_loop = bs // tile_b
    pypto.set_vec_tile_shapes(1, 4, 1, 64)

    for idx in pypto.loop(0, b_loop, 1, name="LOOP_L0_bIdx", idx_name="idx"):
        b_offset = idx * tile_b
        b_offset_end = ((idx + 1) * tile_b).min(bs)
        input_view = pypto.view(input_tensor, [tile_b, N1, N2, DIM], [b_offset, 0, 0, 0],
                                valid_shape=[b_offset_end - b_offset, N1, N2, DIM])
        softmax_out = acl_softmax_core(input_view)
        output_tensor[b_offset:b_offset_end, ...] = softmax_out



### 5.1 PyTorch 图里的自定义 Softmax

PyPTO kernel 想进入 `torch.compile` 和图捕获流程，通常需要再包一层普通 Python 函数。这个包装函数负责两件事：给 kernel 准备输出 Tensor，并处理 FakeTensor 分支。

这里的逻辑可以拆成三步：

1. 如果当前是编译跟踪阶段，返回一个形状、dtype、device 都正确的占位 Tensor。
2. 如果是正常执行阶段，先分配输出 Tensor。
3. 调用 `acl_softmax_kernel` 把结果写入输出 Tensor，再返回它。

`FakeTensor` 分支之所以不能真正执行 PyPTO kernel，是因为它只提供 shape / dtype / device 这类元信息，不代表真实可写的设备缓冲区。这里返回 `torch.zeros(x.shape, dtype=x.dtype, device=f'{x.device}')`，是为了让编译器继续推导图的形状和类型。

`dynamic=True` 这个参数在例子里主要是为了保留动态图入口，让模块签名和后续调用路径保持一致；真正的动态 shape 仍然由输入 Tensor 的实际 shape 决定。

所以这一层包装不是业务逻辑本身，而是让自定义 kernel 能顺利参与图编译、图推导和图捕获。


In [ ]:
if allow_in_graph is not None:
    @allow_in_graph
    def acl_softmax(x: torch.Tensor, dynamic: bool = True) -> torch.Tensor:
        if FakeTensor is not None and isinstance(x, FakeTensor):
            return torch.zeros(x.shape, dtype=x.dtype, device=f"{x.device}")
        output_tensor = torch.empty(x.shape, dtype=x.dtype, device=f"{x.device}")
        acl_softmax_kernel(x, output_tensor)
        return output_tensor
else:
    def acl_softmax(x: torch.Tensor, dynamic: bool = True) -> torch.Tensor:
        output_tensor = torch.empty(x.shape, dtype=x.dtype, device=f"{x.device}")
        acl_softmax_kernel(x, output_tensor)
        return output_tensor


class SoftmaxModule(torch.nn.Module):
    def forward(self, x, dynamic=True):
        return acl_softmax(x, dynamic)


In [ ]:
def test_softmax_capture(device_id=None, dynamic: bool = True) -> None:
    if torch_npu is None:
        print("torch_npu is not available; skip ACLGraph execution.")
        return

    if device_id is None:
        device_id = torch.npu.current_device()
    else:
        torch.npu.set_device(device_id)

    shape = (32, N1, N2, DIM)
    x = torch.rand(shape, dtype=torch.float32, device=f"npu:{device_id}")
    model = torch.compile(SoftmaxModule(), backend="eager", dynamic=True)

    g = torch.npu.NPUGraph()
    with torch.npu.graph(g):
        y = model(x, dynamic)

    g.replay()
    torch.npu.synchronize()
    golden = torch.softmax(x, dim=-1).cpu()

    print(f"ACLGraph input shape: {x.shape}")
    print(f"ACLGraph output shape: {y.shape}")
    assert_allclose(y.detach().cpu().numpy(), golden.detach().cpu().numpy(), rtol=3e-3, atol=3e-3)



### 5.2 捕获与重放

ACLGraph 需要真实 NPU 环境。没有 NPU 时，可以先重点理解代码路径：构造输入、`torch.compile`、进入 `torch.npu.graph(g)` 捕获、再用 `g.replay()` 重放。

要把这条链路看顺，顺序可以记成：

1. 准备 NPU 上的输入 Tensor。
2. 用 `torch.compile(SoftmaxModule(), backend="eager", dynamic=True)` 包装模块。
3. 创建 `torch.npu.NPUGraph()`。
4. 在 `with torch.npu.graph(g):` 中执行一次前向，完成 capture。
5. 调用 `g.replay()` 重放已捕获图。
6. `torch.npu.synchronize()` 后再做数值对比。

这里和 Cost Model 的最大区别是：Cost Model 看的是分析文件是否生成，ACLGraph 看的是捕获和重放是否成功。

需要注意，图捕获期间不要插入会破坏 capture 流程的同步、调试或额外状态操作。同步和验证应该放在 replay 之后。


In [ ]:
test_softmax_capture()



## 6. Cost Model 与 ACLGraph 的区别

| 维度 | Cost Model | ACLGraph |
| --- | --- | --- |
| 主要目标 | 评估和观察执行成本 | 捕获并复用执行图 |
| 运行环境 | 通常可在 SIM / CPU 环境观察输出 | 需要真实 NPU |
| 典型输出 | `merged_swimlane.json` 等分析文件 | 可 replay 的 NPU graph |
| 关注重点 | 硬件单元、任务调度、瓶颈分析 | Host 侧提交开销、图捕获、重复执行效率 |
| 验证方式 | 数值正确 + 输出文件存在且可解析 | 数值正确 + capture / replay 成功 |

它们不是互相替代的关系。Cost Model 更像放大镜，帮助理解算子的执行形态；ACLGraph 更像缓存层，帮助减少重复调度成本。两者结合起来，才是“既看懂，又跑快”的完整系统视角。



## 7. 本节 API 速览

| API / 对象 | 作用 |
| --- | --- |
| `pypto.RunMode.SIM` | 在模拟模式下运行 Cost Model |
| `safe_json_load` | 安全读取分析 JSON，避免文件缺失或格式错误直接中断 |
| `get_output_path` | 定位最新的 `./output` 目录 |
| `pypto.loop` | 将 batch 维度拆成显式循环 |
| `pypto.view` / 切片 | 生成当前 batch 的局部视图 |
| `pypto.assemble` | 将局部结果写回输出张量 |
| `merged_swimlane.json` | Cost Model 的可视化分析输入之一 |
| `@allow_in_graph` | 允许自定义函数进入 `torch.compile` 图 |
| `FakeTensor` | 编译跟踪阶段的占位 Tensor 类型 |
| `torch.compile` | 编译包含 PyPTO 自定义函数的模块 |
| `torch.npu.NPUGraph` | 创建 NPU graph |
| `torch.npu.graph(g)` | 捕获图执行 |
| `g.replay()` | 重放已捕获的图 |
| `torch.npu.synchronize()` | 等待 NPU 执行完成，便于后续验证 |

这一节的 API 更偏系统集成，不是在学新的数学算子，而是在学算子如何进入分析链路和图执行链路。


## 8. 课后练习

本节练习用于复盘 Cost Model 和 ACLGraph 的作用。请结合 Softmax cost model 示例和 ACLGraph capture/replay 示例完成以下题目。

1. （选择题）Cost Model 为什么适合用 Softmax 这类算子观察？  
   A. Softmax 同时包含 reduction、elementwise、loop 和输出写回，执行路径较典型  
   B. Softmax 不需要任何 Tensor 输入  
   C. Softmax 会改变 Cost Model 的数学公式  
   D. Softmax 只能在 CPU 上运行
2. （填空题）`merged_swimlane.json` 主要用来分析________。
3. （选择题）ACLGraph 为什么需要 `@allow_in_graph`？  
   A. 避免自定义包装函数在 PyTorch 图捕获过程中被断开  
   B. 强制所有 Tensor 变成 FakeTensor  
   C. 删除 NPU graph replay  
   D. 改变 Softmax 输出
4. （选择题）FakeTensor 分支为什么不能真正执行 PyPTO kernel？  
   A. FakeTensor 只有元信息，没有真实可执行的数据缓冲区  
   B. FakeTensor 一定没有 shape  
   C. FakeTensor 只能保存字符串  
   D. FakeTensor 会自动执行 NPU kernel
5. （填空题）`graph capture` 发生在________；`replay` 发生在________。

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/04.06_answer.txt



## 9. 本节小结

到这里，你已经完成了系统分析与加速部分：Cost Model 帮助观察算子的模拟执行成本，ACLGraph 帮助捕获并复用执行图。至此，从基础算子组合、归一化和 FFN、动态 shape 和控制流，到 Attention / Transformer 组合和系统优化视角，已经形成一条完整的 PyPTO 中高级实践路线。

如果前几节回答的是“怎么写”，那么这一节回答的就是“怎么分析、怎么复用、怎么跑得更稳”。
